In [1]:
from utils import simulation, calculate_true_survival_probability, create_weibull_data, stratified_split, create_data_with_ipc_weights
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import os
import hashlib
import psutil
import time
from sksurv.util import Surv
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from lifelines import KaplanMeierFitter, WeibullAFTFitter
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv
from sklearn.model_selection import train_test_split
from class_DecisionTreeBaggingClassifier import DecisionTreeBaggingClassifier


### Simulations Parameter ##############
seed = 42
n_sim = 10
B_1  = 3

jk_ab_calc = True
boot_var_calc = True
ijk_std_calc = True
train_models = True

### Weibull daten erstellen

In [41]:
n=      10
seed =  43
tau = 20

params =   { 'shape_weibull': 1,  'p_1': -0.405, 'p_2': -0.4, 'p_3': -0.05, 'p_4': -0.01, 'n': n,
                                'scale_weibull_base':   5597.308204063027       , 
                                'rate_censoring':       0.038465201478012315    ,          }

data = create_weibull_data(params=params, random_state=seed)
df_train, df_test = stratified_split(data, random_state=seed)

df_train = create_data_with_ipc_weights(data=df_train, tau=tau)
df_train

,X_1,X_2,X_3,X_4,event,time,survived,weights_ipcw
0,0,1,80.092751,41.689356,True,6.293821,0,0.166667
1,0,0,64.320841,34.140011,False,34.103429,1,0.166667
2,0,0,59.493906,42.204870,True,20.781967,1,0.166667
3,0,1,72.742805,41.981463,True,23.764004,1,0.166667
4,0,0,59.221315,33.879919,False,0.169456,999,0.000000
5,0,1,80.701150,40.129855,False,23.805986,1,0.166667
6,1,0,89.817201,30.944711,True,27.866978,1,0.166667


### Generierung der Ereigniszeiten/Zensierzeiten basierend auf der Weibull-/ZensierVerteilung aus params 

In [ ]:
### cut data at tau // create ipcw weights ###################################################################
df_train = create_new_dataset_with_ipcw_weights(data=df_train, t=tau)
df_test = create_new_dataset_with_ipcw_weights(data=df_test, t=tau)

### stats for training data and test data ####################################################################
# calculate portion of events and censored data after cut for training data
portions_at_cutpoint = df_train["survived"].value_counts(normalize=True)
if 999 in portions_at_cutpoint.keys():
    portion_censored_after_cut_train = portions_at_cutpoint[999]
else:
    portion_censored_after_cut_train = 0

if 0 in portions_at_cutpoint.keys():
    n_events_after_cut_train = portions_at_cutpoint[0] * df_train.shape[0]
else:
    n_events_after_cut_train = 0

# calculate portion of events and censored data after cut  for test data
portions_at_cutpoint_test = df_test["survived"].value_counts(normalize=True)

if 999 in portions_at_cutpoint_test.keys():
    portion_censored_after_cut_test = portions_at_cutpoint_test[999]
else:
    portion_censored_after_cut_test = 0

if 0 in portions_at_cutpoint_test.keys():
    n_events_after_cut_test = portions_at_cutpoint_test[0] * df_test.shape[0]
else:
    n_events_after_cut_test = 0